# Export LUCAS LUC Land Cover for Lithuania

Exports:
- **PNG rasters** (dashboard style)
- **GeoTIFF rasters** (`rasters/lucas/geotiff/`) for web display – zoomable
- **CSV** per-class area counts

**Fixes:**
- Standard 5-class order: 1=Water, 2=Wetland, 3=Urban, 4=Agriculture, 5=Forest (matches dashboard)
- flipud for correct north-up display (lat ascending → row 0=south, flip for GeoTIFF)

Run all cells. Use a kernel where rasterio works for GeoTIFF export.

In [1]:
from pathlib import Path
import json
from shapely.geometry import shape, Point
from shapely.ops import unary_union
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.transform import from_bounds
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

BASE = Path(r"C:\Users\matas\Desktop\LEI\Data")
LT_GEOJSON = BASE / "lt_boundary_admin.json"
LUCAS_PATTERN = BASE / "Lucas_Luc" / "LUC_hist_EU_afts_v1.1_1-7" / "LUCAS_LUC_v1.1_historical_Europe_0.1deg_*.nc"
OUT_RASTERS = BASE / "rasters" / "lucas"
OUT_GEOTIFF = OUT_RASTERS / "geotiff"
OUT_CSV = BASE / "outputs" / "lucas_lithuania_timeseries.csv"
OUT_RASTERS.mkdir(parents=True, exist_ok=True)
OUT_GEOTIFF.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with open(LT_GEOJSON, "r", encoding="utf-8") as f:
    lt_data = json.load(f)
geoms = [shape(feat["geometry"]) for feat in lt_data["features"]]
lt_geom = unary_union(geoms)

def polygon_mask(lat_vals, lon_vals, geom):
    h, w = len(lat_vals), len(lon_vals)
    mask = np.zeros((h, w), dtype=bool)
    for i, lat in enumerate(lat_vals):
        for j, lon in enumerate(lon_vals):
            mask[i, j] = geom.contains(Point(lon, lat))
    return mask

LAT_MIN, LAT_MAX = 53.5, 56.6
LON_MIN, LON_MAX = 20.5, 26.7

# Standard dashboard order: 1=Water, 2=Wetland, 3=Urban, 4=Agriculture, 5=Forest
group_to_lctypes = {
    "Water": [],
    "Wetland": [12],
    "Urban": [15],
    "Agriculture": [13, 14],
    "Forest": [3, 4, 5, 6],
}
group_names = list(group_to_lctypes.keys())
colors = ["#4DA6FF", "#7B68EE", "#FF4D4D", "#FFD24D", "#228B22"]
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = matplotlib.colors.Normalize(vmin=1, vmax=5)

print("Opening LUCAS LUC datasets matching:", LUCAS_PATTERN)
ds = xr.open_mfdataset(str(LUCAS_PATTERN), chunks="auto", combine="by_coords")
frac = ds["landCoverFrac"]
sub = frac.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
lat_vals = sub["lat"].values
lon_vals = sub["lon"].values
mask = polygon_mask(lat_vals, lon_vals, lt_geom)

time_vals = frac["time"].values
years_all = pd.to_datetime(time_vals).year.astype(int)
years = np.unique(years_all)
print("Exporting LUCAS years:", years)

Opening LUCAS LUC datasets matching: C:\Users\matas\Desktop\LEI\Data\Lucas_Luc\LUC_hist_EU_afts_v1.1_1-7\LUCAS_LUC_v1.1_historical_Europe_0.1deg_*.nc
Exporting LUCAS years: [1950 1951 1952 1953 1954 1955 1956 1957 1958 1959 1960 1961 1962 1963
 1964 1965 1966 1967 1968 1969 1970 1971 1972 1973 1974 1975 1976 1977
 1978 1979 1980 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1991
 1992 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005
 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015]


In [2]:
records = []
for year in years:
    layer = sub.sel(time=str(year), method="nearest")
    group_fracs = []
    for name in group_names:
        codes = group_to_lctypes[name]
        if codes:
            gf = layer.sel(lctype=codes).sum(dim="lctype")
        else:
            gf = 0 * layer.isel(lctype=0).drop_vars("lctype", errors="ignore")
        group_fracs.append(gf)
    stack = xr.concat(group_fracs, dim="group").assign_coords(group=np.arange(len(group_names)))
    maxfrac = stack.max(dim="group")
    dominant = stack.argmax(dim="group") + 1
    arr = dominant.values.astype("float32")
    arr = np.where(np.isfinite(maxfrac.values), arr, np.nan)
    arr_masked = np.where(mask, arr, np.nan)
    arr_display = np.flipud(arr_masked)  # lat ascending: row 0=south, flip for north-up

    flat = arr_masked[np.isfinite(arr_masked)].astype(int)
    if flat.size:
        uniq, cnts = np.unique(flat, return_counts=True)
        for cls_id, cnt in zip(uniq, cnts):
            cls_id = int(cls_id)
            cls_name = group_names[cls_id - 1] if 1 <= cls_id <= len(group_names) else f"class_{cls_id}"
            records.append((int(year), cls_id, cls_name, int(cnt)))

    rgba = cmap(norm(arr_display))
    out_png = OUT_RASTERS / f"lucas_{int(year)}.png"
    plt.imsave(out_png, rgba)

    arr_for_tif = arr_display
    h, w = arr_for_tif.shape
    west, east = float(np.min(lon_vals)), float(np.max(lon_vals))
    south, north = float(np.min(lat_vals)), float(np.max(lat_vals))
    arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
    transform = from_bounds(west, south, east, north, w, h)
    out_tif = OUT_GEOTIFF / f"lucas_{int(year)}.tif"
    with rasterio.open(out_tif, "w", driver="GTiff", height=h, width=w, count=1,
                      dtype=arr_uint8.dtype, crs="EPSG:4326", transform=transform, nodata=0) as dst:
        dst.write(arr_uint8, 1)

    if int(year) % 20 == 0:
        print(f"  Saved {out_png}, {out_tif}")

print("Done.")

C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\dask\array\reductions.py:784: RuntimeWarning: All-NaN slice encountered
  vals = func(x, axis=arg_axis, keepdims=True)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\dask\array\reductions.py:784: RuntimeWarning: All-NaN slice encountered
  vals = func(x, axis=arg_axis, keepdims=True)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\dask\array\reductions.py:784: RuntimeWarning: All-NaN slice encountered
  vals = func(x, axis=arg_axis, keepdims=True)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: R

  Saved C:\Users\matas\Desktop\LEI\Data\rasters\lucas\lucas_1960.png, C:\Users\matas\Desktop\LEI\Data\rasters\lucas\geotiff\lucas_1960.tif


C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\dask\array\reductions.py:784: RuntimeWarning: All-NaN slice encountered
  vals = func(x, axis=arg_axis, keepdims=True)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\dask\array\reductions.py:784: RuntimeWarning: All-NaN slice encountered
  vals = func(x, axis=arg_axis, keepdims=True)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\dask\array\reductions.py:784: RuntimeWarning: All-NaN slice encountered
  vals = func(x, axis=arg_axis, keepdims=True)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: R

  Saved C:\Users\matas\Desktop\LEI\Data\rasters\lucas\lucas_1980.png, C:\Users\matas\Desktop\LEI\Data\rasters\lucas\geotiff\lucas_1980.tif


C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\m

  Saved C:\Users\matas\Desktop\LEI\Data\rasters\lucas\lucas_2000.png, C:\Users\matas\Desktop\LEI\Data\rasters\lucas\geotiff\lucas_2000.tif


C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)
C:\Users\m

Done.


C:\Users\matas\AppData\Local\Temp\ipykernel_2872\463826127.py:36: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_for_tif), arr_for_tif.astype(np.uint8), 0)


In [3]:
df = pd.DataFrame(records, columns=["year", "class_id", "class_name", "count"])
df.to_csv(OUT_CSV, index=False)
print("Saved CSV:", OUT_CSV)
df.head(10)

Saved CSV: C:\Users\matas\Desktop\LEI\Data\outputs\lucas_lithuania_timeseries.csv


,year,class_id,class_name,count
0,1950,3,Urban,3
1,1950,4,Agriculture,762
2,1950,5,Forest,142
3,1951,3,Urban,3
4,1951,4,Agriculture,759
5,1951,5,Forest,145
6,1952,3,Urban,3
7,1952,4,Agriculture,759
8,1952,5,Forest,145
9,1953,3,Urban,3
